2.1 
卷积输出尺寸公式：
Hout​=⌊SHin​+2×P−K​⌋+1,Wout​=⌊SWin​+2×P−K​⌋+1 
输入：Cin​=3, Hin​=32, Win​=32      卷积核：K=5，P=2，S=2，输出通道 Cout​=16
1. 特征图尺寸
Hout​=⌊232+2×2−5​⌋+1=⌊231​⌋+1=15+1=16
宽和高计算一致：Wout​=16    输出尺寸：16×16×16（通道 × 高 × 宽）
2. 单个输出像素的乘法次数
卷积核尺寸：Cin​×K×K=3×5×5，每个位置做一次点乘，总乘法数：
3×5×5=75


In [1]:
import numpy as np

def max_pool2d(inputs, kernel_size, stride, padding=0):
    """
    手动实现2D最大池化前向传播
    :param inputs: 输入张量 shape: (N, C, H, W)
    :param kernel_size: 池化核大小 (kh, kw)
    :param stride: 步幅 (sh, sw)
    :param padding: 填充 (ph, pw)
    :return: 池化输出
    """
    if isinstance(kernel_size, int):
        kernel_size = (kernel_size, kernel_size)
    if isinstance(stride, int):
        stride = (stride, stride)
    if isinstance(padding, int):
        padding = (padding, padding)
    
    N, C, H, W = inputs.shape
    kh, kw = kernel_size
    sh, sw = stride
    ph, pw = padding
    
    pad_input = np.pad(inputs, ((0,0), (0,0), (ph,ph), (pw,pw)), mode='constant')
    H_pad, W_pad = pad_input.shape[2], pad_input.shape[3]
    
    out_h = (H_pad - kh) // sh + 1
    out_w = (W_pad - kw) // sw + 1
    
    output = np.zeros((N, C, out_h, out_w))

    for n in range(N):
        for c in range(C):
            for h in range(out_h):
                for w in range(out_w):
                    h_start = h * sh
                    h_end = h_start + kh
                    w_start = w * sw
                    w_end = w_start + kw
                    window = pad_input[n, c, h_start:h_end, w_start:w_end]
                    output[n, c, h, w] = np.max(window)
    return output

if __name__ == "__main__":
    # 构造测试输入 (N=1, C=1, H=4, W=4)
    x = np.array([[[[1, 2, 3, 4],
                    [5, 6, 7, 8],
                    [9,10,11,12],
                    [13,14,15,16]]]])
    res = max_pool2d(x, kernel_size=2, stride=2, padding=0)
    print("最大池化输出：")
    print(res)

最大池化输出：
[[[[ 6.  8.]
   [14. 16.]]]]


3.1 
卷积层参数量：Cin​×Cout​×Kh​×Kw​
1. 单个 5×5 卷积层参数量
Params1​=C×C×5×5=25C2
2. 两层串联 3×3 卷积总参数量
第一层：C×C×3×3=9C2 第二层：C×C×3×3=9C2 合计：
Params2​=9C2+9C2=18C2


In [2]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super().__init__()
        self.nin_block = nn.Sequential(
            # 第一层普通卷积 + ReLU
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, 
                      stride=stride, padding=padding),
            nn.ReLU(),
            # 第一个1x1卷积 + ReLU
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            # 第二个1x1卷积 + ReLU
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.nin_block(x)

# 测试
if __name__ == "__main__":
    block = NiNBlock(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
    x = torch.randn(1, 3, 32, 32)
    y = block(x)
    print("NiN Block 输出形状:", y.shape)

NiN Block 输出形状: torch.Size([1, 16, 32, 32])


4.1 批量归一化
已知：x1​=2, x2​=4, x3​=6, x4​=8, γ=2, β=1, ϵ=0
均值： μ=(2+4+6+8)/4=5
方差： σ2=[(2−5)2+(4−5)2+(6−5)2+(8−5)2]/4=(9+1+1+9)/4=5
BN 公式：x^i​=(xi​−μ)/σ2+ϵ​,  yi​=γ⋅x^i​+β
y1​=2∗(2−5)/5​+1=1−65​/5 y2​=2∗(4−5)/5​+1=1−25​/5 y3​=2∗(6−5)/5​+1=1+25​/5 y4​=2∗(8−5)/5​+1=1+65​/5


In [3]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super().__init__()
        # 主分支：两个3x3卷积 + BN
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               padding=1, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, 
                               padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        #  shortcut分支：可选1x1卷积调整通道/尺寸
        if use_1x1conv:
            self.shortcut = nn.Conv2d(in_channels, out_channels, 
                                     kernel_size=1, stride=stride)
        else:
            self.shortcut = nn.Identity()
        
        self.relu = nn.ReLU()

    def forward(self, x):
        # 主路径
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        # 残差连接
        out += self.shortcut(x)
        out = self.relu(out)
        return out

# 测试
if __name__ == "__main__":
    # 不使用1x1卷积
    res1 = Residual(64, 64, use_1x1conv=False)
    x = torch.randn(1, 64, 32, 32)
    print("无1x1卷积输出形状:", res1(x).shape)
    
    # 使用1x1卷积升维+改步幅
    res2 = Residual(64, 128, use_1x1conv=True, stride=2)
    print("有1x1卷积输出形状:", res2(x).shape)

无1x1卷积输出形状: torch.Size([1, 64, 32, 32])
有1x1卷积输出形状: torch.Size([1, 128, 16, 16])


5.1 
 底层小学习率、顶层大学习率的原因
1.预训练模型底层卷积学习通用视觉特征（边缘、纹理、轮廓），这类特征具有普适性，无需大幅更新，因此使用小学习率或冻结参数，避免破坏已学到的有效特征。
2.顶层全连接 / 分类层是针对原数据集的分类逻辑，目标任务完全不同，参数需要从头适配，因此使用大学习率快速收敛。
 数据集小且与源数据集相似的微调策略
冻结全部底层特征层，仅训练最后分类层；
若效果不足，再极小范围放开顶层少量层，并使用极小学习率；
配合图像增广、权重衰减、Dropout 进一步抑制过拟合；
不使用全局大学习率，防止小数据集过拟合。

In [4]:
from torchvision import transforms

# 构建图像增广Pipeline
train_transform = transforms.Compose([
    # 随机裁剪+缩放到224×224，面积比例0.08~1.0
    transforms.RandomResizedCrop(size=224, scale=(0.08, 1.0)),
    # 50%概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 随机亮度、对比度、饱和度，变化系数0.5
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    # 转为Tensor
    transforms.ToTensor()
])



6.1
真实框 A=[10,10,50,50]，预测框 B=[30,30,70,70]
交集左上角：x1=max (10,30)=30，y1=max (10,30)=30
交集右下角：x2=min (50,70)=50，y2=min (50,50)=50
交集面积：S_inter = (50-30)(50-30) = 2020 = 400
A 框面积：S_A = (50-10)(50-10) = 4040 = 1600
B 框面积：S_B = (70-30)(70-30) = 4040 = 1600
并集面积：S_union = S_A + S_B - S_inter = 1600 + 1600 - 400 = 2800
IoU = S_inter / S_union = 400 / 2800 = 1/7

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, labels, num_classes, eps=0.1):
    """
    标签平滑交叉熵损失
    :param logits: 模型原始输出 (N, num_classes)
    :param labels: 真实标签 (N,) 类别索引
    :param num_classes: 分类总数 K
    :param eps: 平滑系数 ε
    :return: 损失值
    """
    # 平滑后目标分布
    soft_target = torch.full_like(logits, fill_value=eps / (num_classes - 1))
    # 真实标签位置赋值 1-eps
    soft_target.scatter_(dim=1, index=labels.unsqueeze(1), value=1 - eps)
    # 计算log_softmax
    log_prob = F.log_softmax(logits, dim=1)
    # 交叉熵 = -sum(soft_target * log_prob)
    loss = -torch.sum(soft_target * log_prob, dim=1).mean()
    return loss

# 测试
if __name__ == "__main__":
    K = 10  # 10分类
    batch_logits = torch.randn(8, K)   # 8个样本
    batch_labels = torch.randint(0, K, (8,))
    loss = label_smoothing_cross_entropy(batch_logits, batch_labels, K, eps=0.1)
    print("标签平滑交叉熵损失:", loss.item())

标签平滑交叉熵损失: 3.1279120445251465
